In [ ]:
import pandas as pd
import numpy as np

from src.testing_validation.model_test import calculate_rmse
from src.helpers import fit_cv_timeseries_model

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

All Data

In [ ]:

combined_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)

combined_data["usd_zar_28_movement"] = (
    combined_data["usd_zar_28"] - combined_data["usd_zar"]
)
combined_data.head(1)

In [ ]:
X_all = combined_data.drop(columns=["date", "usd_zar", "usd_zar_28_movement"])

In [ ]:
y = combined_data["usd_zar_28_movement"]

In [ ]:
model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
)
fit_cv_timeseries_model(model, X_all, y)

Engineered data

In [ ]:

engineered_data = combined_data.copy().drop(columns=["date", "usd_zar_28", "usd_zar_28_movement"])
engineered_data["interest_rate_diff"] = (
    engineered_data["sa_repo_rate"] - engineered_data["us_fed_funds"]
)
engineered_data["sa_us_5y_yield_spread"] = (
    engineered_data["sa_5y_yield"] - engineered_data["us_5y_yield"]
)

engineered_data["commodities"] = np.mean([engineered_data.iron_ore_usd_per_tonne, 
engineered_data.gold_usd_per_oz, engineered_data.platinum_usd_per_oz, engineered_data.richards_bay_coal_usd], axis=0)

engineered_features = engineered_data.columns
# [
#     #"commodities",
#     #"usd_zar",
#     #"gold_usd_per_oz",
#     #"platinum_usd_per_oz",
#     #"richards_bay_coal_usd",
#     #"iron_ore_usd_per_tonne",
#     "brent_usd_per_barrel",
#     "interest_rate_diff",
#     "sa_us_5y_yield_spread",
#     "sa_yoy_inflation",
#     "sa_5y_cds_bp",
#     "vix",
#     "broad_usd_index",
#     "sa_cpi",
#     #"gold_usd_per_oz_return",
#     #"platinum_usd_per_oz_return",
#     #"usd_zar_1w_return",
#     #"usd_zar_1m_return",
#     #"usd_zar_3m_return",
#     #"usd_zar_1m_volatility",
# ]

X_engineered = engineered_data[engineered_features]
assert X_engineered.select_dtypes(exclude="number").empty

In [ ]:
model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
)
fit_cv_timeseries_model(model, X_engineered, y)

Backward stepwise feature selection

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, cross_val_score


def backward_stepwise_selection(estimator, X, y, min_features=1, tolerance=0.0):
    """Remove one feature at a time when doing so improves time-series CV RMSE."""
    selected_features = list(X.columns)
    cv = TimeSeriesSplit()

    def cv_rmse(features):
        scores = cross_val_score(
            clone(estimator),
            X[features],
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1,
        )
        return -scores.mean()

    current_rmse = cv_rmse(selected_features)
    history = [{"removed": None, "n_features": len(selected_features), "cv_rmse": current_rmse}]

    while len(selected_features) > min_features:
        candidates = []
        for feature in selected_features:
            remaining = [name for name in selected_features if name != feature]
            candidates.append((cv_rmse(remaining), feature))

        candidate_rmse, feature_to_remove = min(candidates)
        if candidate_rmse > current_rmse - tolerance:
            break

        selected_features.remove(feature_to_remove)
        current_rmse = candidate_rmse
        history.append(
            {
                "removed": feature_to_remove,
                "n_features": len(selected_features),
                "cv_rmse": current_rmse,
            }
        )
        print(f"Removed {feature_to_remove}; CV RMSE: {current_rmse:.6f}")

    return selected_features, pd.DataFrame(history)


In [ ]:
selection_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(64, 32),
        alpha=0.001,
        max_iter=1000,
        early_stopping=True,
        random_state=42,
    ),
)
best_features, selection_history = backward_stepwise_selection(
    selection_model,
    X_engineered,
    y,
)

print(f"Selected {len(best_features)} of {X_engineered.shape[1]} features")
print(best_features)
selection_history

Regularized multi-layer perceptron with selected features

In [ ]:
regularized_mlp_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        solver="adam",
        alpha=0.01,
        learning_rate="adaptive",
        learning_rate_init=0.001,
        batch_size=64,
        max_iter=2000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=42,
    ),
)
fit_cv_timeseries_model(regularized_mlp_model, X_engineered[best_features], y)